# Final test: the fine-tuned selector on the untouched test split

Every decision so far - 512 px, one epoch, the routing rule, the blank-image control - was
made while looking at the 406 validation rows. This notebook scores the same adapter, with the
same prompt and processor, **once**, on the 1,256 depth/width/height rows of the test split
(one row dropped: its photo also appears in training under another group_id).

Nothing here is tuned. It also saves the model's probability over the candidate letters for
every row, so a confidence-based router can later be fitted on **validation** rows and checked
here without another GPU run.


In [ ]:
import glob, os, subprocess, sys, json, math, re, string, time
from collections import Counter

MODEL_ID   = "Qwen/Qwen2-VL-2B-Instruct"
MAX_PIXELS = 512 * 512           # identical to training

found = glob.glob("/kaggle/input/**/test.csv", recursive=True)
if not found:
    raise SystemExit("test.csv not found - attach the dim-test-split dataset")
DEST = os.path.dirname(found[0])
cfg = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
if not cfg:
    raise SystemExit("adapter_config.json not found - attach the nlp-sem-ft notebook output")
ADAPTER = os.path.dirname(cfg[0])
WORK = "/kaggle/working"
print("data   :", DEST, "|", len(os.listdir(f"{DEST}/images")), "images")
print("adapter:", ADAPTER, sorted(os.listdir(ADAPTER)))

for _k in ("PYTORCH_CUDA_ALLOC_CONF", "PYTORCH_ALLOC_CONF"):
    os.environ[_k] = "expandable_segments:True"
import torch
print("cuda", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
if not torch.cuda.is_available():
    raise SystemExit("No GPU")


In [ ]:
# Same repair as the training run: Kaggle's torchao 0.10.0 breaks transformers 5.x on import.
from importlib.metadata import PackageNotFoundError, version as _v
import packaging.version as V
def ver(p):
    try: return _v(p)
    except PackageNotFoundError: return None
tv, tao, pv = ver("transformers"), ver("torchao"), ver("peft")
print("before:  transformers", tv, "| torchao", tao, "| peft", pv)
if tao and tv and V.parse(tao) < V.parse("0.16.0") and V.parse(tv) >= V.parse("4.50"):
    subprocess.run([sys.executable, "-m", "pip", "-q", "uninstall", "-y", "torchao"], check=False)
if pv is None:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "--no-deps", "peft"], check=True)
print("after:   transformers", ver("transformers"), "| torchao", ver("torchao"), "| peft", ver("peft"))
import transformers, peft
print("imports OK")


In [ ]:
import csv
from PIL import Image
sys.path.insert(0, DEST)
import normalize as N

# Verbatim from the training notebook - the prompt must match character for character.
PHRASE = {"depth": "depth (front to back)",
          "width": "width (left to right)",
          "height": "height (top to bottom)"}
SELECT = ("This product photograph shows measurements. Text recognition read "
          "these values from the image:\n{options}\n"
          "Which one is the product's {phrase}?\n"
          "Answer with only the letter.")

def parse_cands(s):
    out = []
    for part in (s or "").split(";"):
        m = re.match(r"\s*(-?\d+(?:\.\d+)?)\s+(\S+)\s*$", part)
        if m:
            u = N.normalize_unit(m.group(2))
            if u:
                out.append((float(m.group(1)), u))
    return out

def render(row, order):
    opts = "\n".join(f"  ({string.ascii_lowercase[j]}) {row['cands'][k][0]:g} {row['cands'][k][1]}"
                      for j, k in enumerate(order))
    return SELECT.format(options=opts, phrase=PHRASE[row["attribute"]])

def dim_ok(pred, lo, hi, gu, tol=0.01):
    if pred is None: return False
    v, u = pred
    if N.family(u) != N.family(gu): return False
    a = N.to_base(v, u)
    return N.to_base(lo, gu)*(1-tol) <= a <= N.to_base(hi, gu)*(1+tol)

rows = []
for r in csv.DictReader(open(f"{DEST}/test.csv")):
    r["cands"] = parse_cands(r["candidates"]); r["gold_index"] = int(r["gold_index"])
    rows.append(r)
print(f"test rows {len(rows):,} | by candidates:",
      dict(sorted(Counter(min(len(r['cands']), 2) for r in rows).items())), "(2 = 2+)")


In [ ]:
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from peft import PeftModel

proc = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=256*28*28, max_pixels=MAX_PIXELS)
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
base = Qwen2VLForConditionalGeneration.from_pretrained(MODEL_ID, dtype=dtype, attn_implementation="sdpa").to("cuda")
model = PeftModel.from_pretrained(base, ADAPTER).eval()
model.config.use_cache = True
n_lora = sum(p.numel() for n, p in model.named_parameters() if "lora_" in n)
print(f"LoRA parameters loaded: {n_lora:,}  (training reported 18,464,768)")
assert n_lora == 18_464_768, "adapter did not load as trained"

# The adapter answers with a bare letter, so the first generated token is the decision.
LETTER_IDS = {}
for L in "abcdefgh":
    ids = proc.tokenizer.encode(L, add_special_tokens=False)
    assert len(ids) == 1, f"letter {L} is not a single token: {ids}"
    LETTER_IDS[L] = ids[0]

def ask(img, prompt, n):
    """Greedy answer exactly as in training eval, plus p(letter) over the n shown options."""
    msgs = [{"role":"user","content":[{"type":"image"},{"type":"text","text":prompt}]}]
    chat = proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = proc(text=[chat], images=[img], return_tensors="pt").to("cuda")
    with torch.no_grad():
        gen = model.generate(**inp, max_new_tokens=6, do_sample=False,
                             output_scores=True, return_dict_in_generate=True)
    raw = proc.batch_decode(gen.sequences[:, inp["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()
    logp = torch.log_softmax(gen.scores[0][0].float(), -1)
    letters = "abcdefgh"[:n]
    lp = torch.stack([logp[LETTER_IDS[L]] for L in letters])
    mass = float(lp.exp().sum())                 # how much of the distribution is on valid letters
    probs = torch.softmax(lp, -1).tolist()       # renormalised over the shown options
    return raw, [round(p, 5) for p in probs], round(mass, 5)


In [ ]:
out, t0 = [], time.time()
for i, r in enumerate(rows):
    cands = r["cands"]; probs, mass = None, None
    if not cands:
        pred, raw = None, "(no candidates)"
    elif len(cands) == 1:
        pred, raw = cands[0], "(single)"
    else:
        img = Image.open(f"{DEST}/images/{r['image_file']}").convert("RGB")
        raw, probs, mass = ask(img, render(r, list(range(len(cands)))), len(cands))
        m = re.search(r"\b([a-h])\b", raw.lower())
        j = string.ascii_lowercase.index(m.group(1)) if m else None
        pred = cands[j] if (j is not None and j < len(cands)) else None
    out.append({**{k: r[k] for k in ("image_file","attribute","gold_lo","gold_hi","gold_unit","gold_index")},
                "n_cands": len(cands), "raw": raw, "probs": probs, "letter_mass": mass,
                "pred": list(pred) if pred else None,
                "dim": bool(dim_ok(pred, float(r["gold_lo"]), float(r["gold_hi"]), r["gold_unit"]))})
    if (i+1) % 100 == 0:
        el = time.time() - t0
        print(f"  {i+1}/{len(rows)}  dim {100*sum(x['dim'] for x in out)/len(out):.1f}%  "
              f"{el/60:.1f} min  eta {el/(i+1)*(len(rows)-i-1)/60:.1f} min", flush=True)
        json.dump(out, open(f"{WORK}/vlm_test_results.json", "w"), indent=1)
json.dump(out, open(f"{WORK}/vlm_test_results.json", "w"), indent=1)
print(f"done in {(time.time()-t0)/60:.1f} min - saved vlm_test_results.json")


In [ ]:
def wilson(k, n, z=1.96):
    if n == 0: return (0.0, 0.0)
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n); h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return 100*(c-h)/d, 100*(c+h)/d

def line(name, xs):
    k = sum(x["dim"] for x in xs); lo, hi = wilson(k, len(xs))
    print(f"{name:<34}{k:>5}/{len(xs):<5}{100*k/len(xs):6.1f}%  [{lo:.1f}, {hi:.1f}]")

print("TEST SPLIT - fine-tuned selector (VLM alone; routing is scored locally)")
line("all rows", out)
for a in ("depth", "width", "height"):
    line(f"  {a}", [x for x in out if x["attribute"] == a])
line("2+ candidates", [x for x in out if x["n_cands"] >= 2])
line("2+ candidates, gold present", [x for x in out if x["n_cands"] >= 2 and x["gold_index"] >= 0])
print(f"gold among candidates (ceiling): {100*sum(x['gold_index'] >= 0 for x in out)/len(out):.1f}%")
print("validation for comparison: 51.5% all | 70.8% 2+ gold-present")

asked = [x for x in out if x["probs"]]
print(f"\nletter mass on valid options: median {sorted(x['letter_mass'] for x in asked)[len(asked)//2]:.3f}")
agree = sum(max(range(len(x["probs"])), key=x["probs"].__getitem__) ==
            ("abcdefgh".find(x["raw"][:1].lower())) for x in asked)
print(f"argmax of saved probs matches greedy answer: {agree}/{len(asked)}")
print("\nDownload vlm_test_results.json from Output.")
